In [ ]:
import os
import base64
from email.message import EmailMessage
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# If modifying these scopes, delete the file token.json.
# This scope allows us to send emails on your behalf.
SCOPES = ['https://www.googleapis.com/auth/gmail.send']

def get_gmail_service():
    """Authenticates the user and returns the Gmail API service object."""
    creds = None
    # The file token.json stores the user's access and refresh tokens.
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    return build('gmail', 'v1', credentials=creds)

def create_and_send_email():
    """Creates and sends an email using the Gmail API."""
    try:
        # 1. Initialize the Gmail API service
        service = get_gmail_service()

        # 2. Build the email message structure
        message = EmailMessage()
        
        # Configure email headers
        message['To'] = 'recipient@example.com'      # Replace with recipient email
        message['From'] = 'sender@gmail.com'    # Replace with your Gmail
        message['Subject'] = 'Automated Quarterly Update Report'

        # Configure a properly formatted body (Plain text or HTML)
        email_body = (
            "Hello Team,\n\n"
            "Please find the summary of our quarterly performance attached to this pipeline.\n"
            "Key Highlights:\n"
            " - Automation pipeline is officially active.\n"
            " - Google API integrations are fully operational.\n\n"
            "Best regards,\n"
            "Automated Python System"
        )
        message.set_content(email_body)

        # Alternatively, if you want to send an HTML formatted body, uncomment below:
        # message.add_alternative("""\
        # <html>
        #   <body>
        #     <p>Hello Team,<br><br>
        #        Please find the summary of our quarterly performance active.<br>
        #        <strong>Key Highlights:</strong>
        #        <ul>
        #          <li>Automation pipeline is active.</li>
        #          <li>Google API integrations operational.</li>
        #        </ul>
        #     </p>
        #   </body>
        # </html>
        # """, subtype='html')

        # 3. Encode the message to base64url string as required by Gmail API
        encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()
        
        create_message = {
            'raw': encoded_message
        }

        # 4. Send the email via Gmail API
        print("Sending email...")
        send_message = (service.users().messages().send(userId="me", body=create_message).execute())
        
        print(f"Message sent successfully! Message ID: {send_message['id']}")

    except HttpError as error:
        print(f"An error occurred: {error}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == '__main__':
    create_and_send_email()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=234069195326-l7s9aa1jr0hr0vma0mbrhu0lgf3ieq46.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A57684%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send&state=7wWi56a10ji7YrS55leODDF3sGN9aW&code_challenge=yr-YNxWRXWRNRp8OpU4vkPaCeatvCl8ZPKgLPtlwugo&code_challenge_method=S256&access_type=offline
Sending email...
Message sent successfully! Message ID: 19e840f8efaf184b
